In [3]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
import torch
from autogluon.tabular import TabularPredictor

# Helper functions defined in your original script
def clean_text(text):
    return " ".join(text.lower().split())

def combine_text(question, answer):
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    return clean_text(q + " " + a)

class ThaiBERTEmbedder:
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def encode(self, text_list, batch_size=16, max_length=128):
        embeddings = []
        for i in range(0, len(text_list), batch_size):
            batch_text = text_list[i : i + batch_size]
            inputs = self.tokenizer(batch_text, padding=True, truncation=True,
                                    max_length=max_length, return_tensors="pt").to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
        return np.vstack(embeddings)

# Load data
train_df = pd.read_csv(os.path.join(REPO_PATH, "data","train.csv"))
test_df = pd.read_csv(os.path.join(REPO_PATH, "data","test.csv"))
submission_df = pd.read_csv(os.path.join(REPO_PATH, "data","sample_submission.csv"))

# Initialize TF-IDF and BERT
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10000)
bert_embedder = ThaiBERTEmbedder()

final_predictions = []

# Train and predict separately for each Q set
for q_set in ["Q1", "Q2", "Q3", "Q4"]:
    print(f"Processing {q_set}...")

    train_subset = train_df[train_df["set"] == q_set]
    test_subset = test_df[test_df["set"] == q_set]

    train_texts = train_subset.apply(lambda x: combine_text(x["question"], x["answer"]), axis=1)
    test_texts = test_subset.apply(lambda x: combine_text(x["question"], x["answer"]), axis=1)

    X_tfidf_train = vectorizer.fit_transform(train_texts).toarray()
    X_tfidf_test = vectorizer.transform(test_texts).toarray()

    X_bert_train = bert_embedder.encode(train_texts.tolist())
    X_bert_test = bert_embedder.encode(test_texts.tolist())

    X_train = np.hstack([X_tfidf_train, X_bert_train])
    X_test = np.hstack([X_tfidf_test, X_bert_test])

    y_train = train_subset["score"].values

    train_data = pd.DataFrame(X_train, columns=[f"f{i}" for i in range(X_train.shape[1])])
    train_data["score"] = y_train

    predictor = TabularPredictor(label="score", problem_type="regression").fit(train_data)

    test_data = pd.DataFrame(X_test, columns=[f"f{i}" for i in range(X_test.shape[1])])
    preds = predictor.predict(test_data)

    preds_df = pd.DataFrame({"ID": test_subset["ID"].values, "score": preds})
    final_predictions.append(preds_df)

# Merge predictions from all Q sets
final_submission = pd.concat(final_predictions).set_index("ID").reindex(submission_df["ID"]).reset_index()

# Ensure no null values in submission
assert final_submission.isnull().sum().sum() == 0, "Null values detected!"

# Save submission
final_submission.to_csv("submission.csv", index=False)
print("Final submission file saved as 'submission.csv'")


Processing Q1...
     f0   f1   f2   f3   f4   f5   f6   f7   f8   f9  ...     f1597     f1598  \
0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.793181  0.332427   
1   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.818094  0.212954   
2   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.812725  0.415516   
3   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.686114  0.249250   
4   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.783022  0.204060   
..  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...       ...       ...   
86  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.795079  0.180538   
87  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.693092  0.303228   
88  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.908785  0.031156   
89  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -0.805991  0.220602   
90  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ... -1.154829  0.329984   

       f15

ValueError: No objects to concatenate